# 트리플 정규화 결과를 그래프에 적재하는 과제 LV2

function_overview, scatterplot, kdeplot, 릴리스 안내의 4개 실제 문서에서 추출한 14개 트리플을 사용합니다.  
검토 기록 1건을 입력에서 제외해, 아직 ID를 확정하지 못한 출현을 보류하는 상황을 연습합니다.  
확정한 표준 ID로 동일 개체를 묶고, 원래 관계와 근거를 보존해 Neo4j에 적재합니다.  
교안 02의 입력 검증, ID 충돌 보류, 노드·관계 보존, 평가와 실제 중복 통합을 연습합니다.  
1~8번은 파일과 파이썬으로 수행합니다. 9번 직전에 실습용 Neo4j에 연결하며, 9~10번에는 APOC가 필요합니다.  


In [ ]:
# [제공코드] JSONL 자료를 읽고 이름과 출현 기록을 비교할 도구를 준비합니다.
import json
from pathlib import Path
from itertools import combinations
from difflib import SequenceMatcher
from pprint import pprint

data_dir = Path("data")

def load_rows(filename):
    """한 줄에 한 기록이 저장된 JSONL 파일을 딕셔너리 목록으로 읽습니다."""
    lines = (data_dir / filename).read_text(encoding="utf-8").splitlines()
    return [json.loads(line) for line in lines if line.strip()]

def pair_key(left_id, right_id):
    """비교 순서가 바뀌어도 같은 두 기록을 같은 키로 나타냅니다."""
    return tuple(sorted((left_id, right_id)))


## 저장된 추출 결과를 살펴봅니다

이 자료는 실제 Seaborn 문서에서 LLM이 추출해 저장한 트리플의 일부입니다.  
원문에서 트리플을 다시 추출하지 않고, 저장된 주어와 목적어를 정리합니다.  
**출현 기록**은 트리플 한 행의 주어 또는 목적어 자리를 따로 기록한 것입니다.  
`triple_id`는 추출 행의 ID, `mention_id`는 그 행의 어느 자리인지 구분하는 출현 ID입니다.  

| 출처 키 | 뜻 |
|---|---|
| source_file | 원본 추출 JSONL 파일 경로 |
| source_line | 그 파일의 행 번호. 1부터 시작 |
| source_triple_index | 해당 행 안의 트리플 번호. 1부터 시작 |

`evidence`와 `source_doc_id`는 원문으로 돌아가 판정 근거를 확인할 때 사용합니다.  


In [ ]:
# [제공코드] kg_lv2_triples.jsonl: 원문과 추출 위치가 보존된 과제용 트리플입니다.
task_triples = load_rows("kg_lv2_triples.jsonl")
triple_by_id = {row["triple_id"]: row for row in task_triples}
original_triples = [dict(row) for row in task_triples]

# kg_corpus.jsonl: 추출의 출처인 실제 문서의 본문과 URL입니다.
task_documents = {row["doc_id"]: row for row in load_rows("kg_corpus.jsonl")}
first_document = task_documents[task_triples[0]["source_doc_id"]]
print("첫 원문:", first_document["title"], first_document["url"])
print(first_document["text"][:500])

print("추출 트리플:", len(task_triples))
for row in task_triples:
    print(row["triple_id"], row["subject"], row["relation"], row["object"])
pprint(task_triples[0])


In [ ]:
# [제공코드] kg_catalog.jsonl: 공식 문서의 API 이름과 문서 URL을 기준으로 만든 프로젝트 개체 목록입니다.
# standard_id는 이 개체 목록에서 고정해 쓰는 표준 ID입니다.
catalog = load_rows("kg_catalog.jsonl")

# kg_alias_review.jsonl: 각 출현의 원문과 타입을 대조해 작성한 ID 연결 판정과 이유입니다.
review_rows = load_rows("kg_alias_review.jsonl")

print("개체 목록:", len(catalog), "/ 출현별 검토 기록:", len(review_rows))


In [ ]:
# [제공코드] 같은 이름이라도 트리플의 양 끝에서 등장한 기록은 각각 보존합니다.
mentions = []
for triple in task_triples:
    for role in ["subject", "object"]:
        mentions.append({
            "mention_id": triple["triple_id"] + ":" + role,
            "triple_id": triple["triple_id"], "role": role,
            "name": triple[role], "entity_type": triple[role + "_type"],
            "source_doc_id": triple["source_doc_id"], "evidence": triple["evidence"],
        })
print("출현 기록:", len(mentions))
pprint(mentions[:2])


In [ ]:
# [제공코드] 보류 처리를 연습하기 위해 첫 출현 기록의 검토 행만 입력에서 제외합니다.
# 원래 트리플과 전체 검토 파일을 수정하지 않습니다.
held_mention_id = mentions[0]["mention_id"]
review_for_task = [row for row in review_rows if row["mention_id"] != held_mention_id]
print("이 과제에서 검토 결과를 제외한 출현 ID:", held_mention_id)


## 1. 연결 입력의 출현 ID와 원본 필드를 검사합니다

**배경**: 다른 추출 버전의 출현 기록을 연결하면 이름이 같아도 잘못된 관계가 만들어집니다.  

**요구사항**  
- **validate_mentions(rows, triples)** 함수를 작성하세요. rows와 triples는 딕셔너리 목록입니다. triples의 각 행에서 주어·목적어의 출현 ID를 만들고, rows가 그 전체를 정확히 한 번씩 포함하는지 검사하세요. rows 순서는 달라도 허용합니다. 필드가 하나라도 빠졌거나 남았으면 값이 맞아도 거부합니다.
- **validate_mentions** 는 각 행의 `mention_id`, `triple_id`, `role`, `name`, `entity_type`, `source_doc_id`, `evidence` 7개 필드가 인자 triples에서 재구성한 출현 기록과 정확히 일치하는지 검사합니다. 누락·중복·범위 밖 ID·값 변경·추가 필드는 ValueError를 발생시키고, 정상이면 True를 반환하세요.
- **input_valid** 에 mentions와 task_triples를 검사한 결과를 담으세요. 에러 메시지는 자유입니다.

**확인 기준**: 정상 입력은 True입니다. 같은 ID의 행을 두 번 넣거나 원문을 변경하면 ValueError입니다.  

<details><summary>힌트</summary>

```text
접근방법:
- ID별 원본 기록을 만들고 전체 범위와 각 행을 대조합니다.

세부구현:
1. 트리플의 두 끝에서 기대하는 출현 기록을 만듭니다.
2. 출현 ID의 중복과 누락을 검사합니다.
3. 해당 ID의 원래 7개 필드를 비교합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert input_valid is True, "정상 입력은 True를 반환하세요."
assert validate_mentions(list(reversed(mentions)), task_triples) is True, "입력 순서 변경은 허용하세요."
bad_inputs = [mentions[:-1], mentions + [dict(mentions[0])]]
for field in mentions[0]:
    changed = [dict(row) for row in mentions]
    changed[0][field] = "검사용 변경 값"
    bad_inputs.append(changed)
extra = [dict(row) for row in mentions]
extra[0]["추가키"] = True
bad_inputs.append(extra)
for rows in bad_inputs:
    rejected = False
    try:
        validate_mentions(rows, task_triples)
    except ValueError:
        rejected = True
    assert rejected, "누락·중복·범위 밖 ID·원본 변경·추가 필드를 거부하세요."
print("✅ 통과!")


다음 문항에서는 교안 02에서 배운 **검토로 표준 ID를 확정하는 단계**를 직접 구현합니다.  
`link_one`의 반환값은 확정한 표준 ID 또는 보류를 뜻하는 `None`입니다.  


## 2. 검토가 끝난 출현 기록만 연결합니다

**배경**: 이름이 같은 다른 출현 기록의 검토를 아직 검토하지 않은 기록에 그대로 적용하면 안 됩니다.  

**요구사항**  
- **link_one(mention, catalog_rows, reviews)** 함수를 작성하세요. catalog_rows의 각 행은 `standard_id`와 `entity_type`을 가집니다. 검토 행의 `mention_id`, `name`, `entity_type`, `source_doc_id`, `evidence`가 mention과 모두 일치하고, 검토 행의 standard_id가 catalog_rows에 존재하고 그 개체의 entity_type이 mention의 entity_type과 같을 때만 그 standard_id를 모으세요.
- **link_one** 은 검토로 확인된 서로 다른 ID가 정확히 하나면 그 문자열을, 없거나 여러 개이면 None을 반환합니다. 같은 ID의 검토 행은 반복되어도 ID 하나로 셉니다.
- **linked** 에는 mentions 중 연결된 기록의 `mention_id: standard_id`를 딕셔너리로 담고, **pending** 에는 나머지 출현 ID를 입력 순서대로 목록에 담으세요. review_for_task를 사용하세요.

**확인 기준**: 28개 출현 중 27개가 연결되고 pending에는 held_mention_id 하나가 남습니다. 자가채점은 문서·근거·타입 불일치, 없는 ID, 서로 다른 ID로 판정한 검토 기록이 섞인 입력도 검사합니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 조건을 모두 만족한 ID를 집합에 모은 뒤 개수로 반환 값을 결정합니다.

세부구현:
1. 개체 목록을 ID로 조회할 수 있게 준비합니다.
2. 검토 행의 다섯 필드와 표준 항목의 타입을 확인합니다.
3. 검토로 확인된 ID가 하나인지 확인하고 전체 출현 기록을 연결과 보류로 나눕니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
review_by_id = {row["mention_id"]: row for row in review_rows}
expected_linked = {row["mention_id"]: review_by_id[row["mention_id"]]["standard_id"] for row in mentions if row["mention_id"] != held_mention_id}
assert linked == expected_linked, "검토되지 않은 출현 기록을 연결하거나 다른 표준 ID를 넣지 마세요."
assert pending == [held_mention_id], "연결하지 못한 출현 ID를 입력 순서로 보존하세요."
probe = mentions[0]
valid_review = dict(review_by_id[probe["mention_id"]])
correct_id = valid_review["standard_id"]
assert link_one(probe, catalog, []) is None, "검토 기록이 없으면 이름이 같아도 보류하세요."
assert link_one(probe, catalog, [valid_review, dict(valid_review)]) == correct_id, "같은 ID의 검토 기록이 반복되어도 ID 하나로 세세요."
for field in ["mention_id", "name", "entity_type", "source_doc_id", "evidence"]:
    changed = dict(valid_review)
    changed[field] = "자가채점용 불일치 값"
    assert link_one(probe, catalog, [changed]) is None, f"검토 기록의 {field}도 일치해야 합니다."
missing = dict(valid_review)
missing["standard_id"] = "자가채점::존재하지않는ID"
assert link_one(probe, catalog, [missing]) is None, "개체 목록에 없는 ID는 연결하지 마세요."
wrong_catalog = [dict(row) for row in catalog]
for row in wrong_catalog:
    if row["standard_id"] == correct_id:
        row["entity_type"] = "자가채점용 다른 타입"
assert link_one(probe, wrong_catalog, [valid_review]) is None, "표준 항목의 타입까지 확인하세요."
another_id = next(row["standard_id"] for row in catalog if row["entity_type"] == probe["entity_type"] and row["standard_id"] != correct_id)
ambiguous = dict(valid_review)
ambiguous["standard_id"] = another_id
assert link_one(probe, catalog, [valid_review, ambiguous]) is None, "검토 기록이 서로 다른 표준 ID를 가리키면 보류하세요."
print("✅ 통과!")


다음 3번은 교안 02의 `check_links`가 다룬 저장 형식 검증입니다. 2번의 원문 검토를 대신하지 않으며, 검사용 복사본만 검사합니다. `stored_links`는 형식 검사 연습용 가상 연결 결과입니다. 미확정 출현은 후보가 없는 상황으로 구성했으며, 실제 후보 검색 결과를 뜻하지 않습니다.  


In [ ]:
# [제공코드] 2번의 확정 결과에 저장용 필드를 붙입니다. 원래 파일과 원본 기록은 수정하지 않습니다.
stored_links = []
for mention in mentions:
    standard_id = linked.get(mention["mention_id"])
    row = dict(mention)
    row.update({"candidate_ids": [standard_id] if standard_id else [],
                "standard_id": standard_id,
                "status": "linked" if standard_id else "unmapped"})
    stored_links.append(row)
print("검사할 저장 기록:", len(stored_links))


## 3. 저장된 연결 기록의 원본과 상태를 검사합니다

**배경**: 저장 파일의 원문은 맞아도 후보와 확정 ID가 서로 모순되면 잘못된 노드를 적재할 수 있습니다.  

**요구사항**  
- **validate_links(mention_rows, link_rows, catalog_rows)** 함수를 작성하세요. 인자는 딕셔너리 목록이며 ID는 문자열입니다. mention_rows는 1번의 원본 7개 필드, catalog_rows는 standard_id와 entity_type을 포함합니다. mention_rows의 mention_id 또는 catalog_rows의 standard_id가 중복되면 ValueError입니다.
- **validate_links** 는 link_rows가 전체 출현 ID를 정확히 한 번씩 포함하는지 검사합니다. 행 순서 변경은 허용하고 누락·중복·범위 밖 ID는 ValueError입니다. 각 연결 행은 원본의 7개 필드와 candidate_ids, standard_id, status의 총 10개 키만 가지며 원본 값이 모두 같아야 합니다.
- **candidate_ids** 는 중복 없는 문자열 목록입니다. 자료형 자체는 검사하지 않아도 되며 순서도 자유입니다. 다만 모든 후보가 catalog_rows에 존재하고 해당 출현과 entity_type이 같아야 합니다.
- **status** 는 linked, review, ambiguous, unmapped 중 하나입니다. linked는 standard_id가 후보 목록에 있어야 합니다(여러 후보 중 검토로 하나를 확정한 경우도 허용). 나머지 세 상태는 standard_id가 None이어야 합니다. review는 후보 1개, ambiguous는 2개 이상, unmapped는 0개여야 합니다.
- **validate_links** 는 하나라도 어기면 ValueError를 발생시키고 정상이면 True를 반환하세요. 에러 메시지는 자유입니다. 세 입력을 수정하지 마세요. **links_valid** 에 mentions, stored_links, catalog를 검사한 반환값을 담으세요.

**확인 기준**: 정상 기록과 순서만 바꾼 기록은 True입니다. 후보가 두 개인 ambiguous 기록은 확정 ID가 None이어야 하며, 후보가 하나인 ambiguous 기록은 거부해야 합니다. 자가채점은 네 상태의 정상 사례와 각 규칙을 어긴 복사본을 검사합니다.  

<details><summary>힌트</summary>

```text
접근방법:
- ID로 원본과 개체 목록을 조회한 뒤 원본 일치와 연결 상태를 나누어 검사합니다.

세부구현:
1. 입력 ID의 중복을 검사하고 원본 및 개체 목록을 ID로 조회할 수 있게 준비합니다.
2. 연결 행의 키와 원본 값, 출현 ID 전체 범위를 확인합니다.
3. 후보 목록의 형식·중복·존재·타입을 확인합니다.
4. 확정 여부와 후보 개수에 맞는 상태인지 확인하고 True를 반환합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert links_valid is True, "정상 저장 기록은 True를 반환하세요."
assert validate_links(mentions, list(reversed(stored_links)), catalog) is True, "행 순서가 달라도 허용하세요."
# 같은 출처의 한 출현을 복사해 네 상태의 정상 경계와 비정상 사례를 구성합니다.
source = mentions[0]
matching_ids = [row["standard_id"] for row in catalog if row["entity_type"] == source["entity_type"]][:3]
wrong_type_id = next(row["standard_id"] for row in catalog if row["entity_type"] != source["entity_type"])
good_cases = [
    ("linked", matching_ids[:1], matching_ids[0]),
    ("linked", list(reversed(matching_ids)), matching_ids[0]),
    ("review", matching_ids[:1], None),
    ("ambiguous", matching_ids[:2], None), ("ambiguous", matching_ids, None), ("unmapped", [], None),
]
for status, candidates, standard_id in good_cases:
    row = dict(source, candidate_ids=list(candidates), standard_id=standard_id, status=status)
    before = json.dumps([[source], [row], catalog], ensure_ascii=False)
    assert validate_links([source], [row], catalog) is True, "상태 규칙을 만족하는 정상 기록을 허용하세요."
    assert json.dumps([[source], [row], catalog], ensure_ascii=False) == before, "검사 과정에서 입력을 수정하지 마세요."
bad_cases = [
    ("linked", matching_ids[:1], matching_ids[1]),
    ("linked", [], None), ("review", [], None),
    ("review", matching_ids, None), ("review", matching_ids[:1], matching_ids[0]),
    ("ambiguous", matching_ids[:1], None), ("ambiguous", matching_ids, matching_ids[0]),
    ("unmapped", matching_ids[:1], None), ("unmapped", [], matching_ids[0]),
    ("unknown", [], None), ("review", ["검사용::없는ID"], None),
    ("review", [wrong_type_id], None), ("ambiguous", [matching_ids[0]] * 2, None),
    ("review", matching_ids[0], None), ("review", [17], None),
]
invalid_rows = [dict(source, status=status, candidate_ids=candidates, standard_id=standard_id)
                for status, candidates, standard_id in bad_cases]
valid_row = dict(source, status="linked", candidate_ids=matching_ids[:1], standard_id=matching_ids[0])
for field in source:
    changed = dict(valid_row)
    changed[field] = "검사용 원본 변경"
    invalid_rows.append(changed)
for field in valid_row:
    changed = dict(valid_row)
    del changed[field]
    invalid_rows.append(changed)
invalid_rows.append(dict(valid_row, extra=True))
invalid_inputs = [([source], [row], catalog) for row in invalid_rows]
invalid_inputs += [([source], [], catalog), ([source], [valid_row, dict(valid_row)], catalog),
                   ([source, dict(source)], [valid_row], catalog),
                   ([source], [valid_row], catalog + [dict(catalog[0])])]
for originals, rows, items in invalid_inputs:
    before = json.dumps([originals, rows, items], ensure_ascii=False)
    rejected = False
    try:
        validate_links(originals, rows, items)
    except ValueError:
        rejected = True
    assert rejected, "원본·후보·상태의 각 규칙을 위반한 저장 기록을 ValueError로 거부하세요."
    assert json.dumps([originals, rows, items], ensure_ascii=False) == before, "잘못된 입력도 검사 중 수정하지 마세요."
print("✅ 통과!")


In [ ]:
# [제공코드] 같다고 확인한 쌍을 그룹으로 모읍니다. 연결되지 않은 기록도 한 개짜리 그룹으로 남깁니다.
def group_pairs(mention_ids, same_pairs):
    """같은 개체로 판정한 쌍을 이어 출현 그룹을 만듭니다.

    Args:
        mention_ids (list[str]): 보존할 전체 출현 ID.
        same_pairs (list[tuple[str, str]]): 같은 개체로 판정한 출현 ID 쌍.

    Returns:
        list[list[str]]: 정렬한 그룹 목록. 연결되지 않은 출현도 단독 그룹으로 남습니다.
    """
    groups = [{mention_id} for mention_id in mention_ids]
    for left_id, right_id in same_pairs:
        # 평가 범위 밖의 ID를 잘못 넣으면 기록이 빠질 수 있으므로 먼저 확인합니다.
        if left_id not in mention_ids or right_id not in mention_ids:
            raise ValueError("동일 판정 쌍에 입력 목록에 없는 ID가 있습니다.")
        joined = set()
        remaining = []
        for group in groups:
            if left_id in group or right_id in group:
                joined.update(group)
            else:
                remaining.append(group)
        remaining.append(joined)
        groups = remaining
    return sorted([sorted(group) for group in groups])

# 입력 예시: 두 초판 표기는 같은 책이고 개정판은 별도 책입니다.
example_ids = ["d01:object", "d02:object", "d03:object"]
example_pairs = [("d01:object", "d02:object")]  # 파이썬 입문과 파이썬 입문서

# 호출: 같은 초판끼리 묶고 개정판(d03:object)은 따로 남깁니다.
print(group_pairs(example_ids, example_pairs))
# 예상 출력: [['d01:object', 'd02:object'], ['d03:object']]


## 4. 대칭성과 이행성을 적용해 동일 개체 그룹을 만듭니다

**배경**: 같은 ID로 확인한 출현 기록들은 비교한 방향과 직접 연결한 순서에 관계없이 한 그룹입니다.  

**요구사항**  
- **id_buckets** 에 linked의 표준 ID를 키로, 해당 출현 ID를 정렬한 목록을 값으로 담으세요.
- **direct_pairs** 에는 각 id_buckets 목록에서 서로 이웃한 두 출현 ID만 pair_key로 정렬한 튜플로 담으세요. 집합을 사용합니다.
- **identity_groups** 에는 전체 mentions의 출현 ID와 direct_pairs를 group_pairs에 전달한 결과를 담으세요. 보류한 출현도 한 개짜리 그룹으로 남깁니다.
- **expanded_pairs** 에는 identity_groups의 각 그룹에서 만들 수 있는 모든 쌍을 집합으로 담으세요. 각 쌍은 pair_key(left_id, right_id)가 반환하는 정렬된 튜플로 표현하세요.
- **symmetry_ok** 에는 모든 direct_pairs에서 pair_key에 두 ID를 반대로 전달해도 같은 튜플인지 나타내는 bool을 담으세요.

**확인 기준**: 그룹은 15개입니다. 직접 연결한 13쌍에서 그룹 내부 전체 30쌍이 만들어집니다. 세 기록 이상인 그룹에서는 직접 연결하지 않은 첫 기록과 마지막 기록의 쌍도 포함됩니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 이웃한 연결로 그룹을 만든 뒤 combinations로 그룹 내부의 모든 쌍을 펼칩니다.

세부구현:
1. 표준 ID별로 출현 ID를 모읍니다.
2. 목록의 이웃한 두 기록을 연결합니다.
3. 전체 출현 ID를 group_pairs에 전달해 보류 기록도 보존합니다.
4. 그룹 내부 모든 쌍과 순서 반전 결과를 확인합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
expected_buckets = {}
for mention_id, standard_id in linked.items():
    expected_buckets.setdefault(standard_id, []).append(mention_id)
expected_buckets = {key: sorted(value) for key, value in expected_buckets.items()}
assert id_buckets == expected_buckets, "표준 ID별 출현 목록을 빠짐없이 정렬하세요."
expected_direct = {pair_key(a, b) for ids in expected_buckets.values() for a, b in zip(ids, ids[1:])}
assert direct_pairs == expected_direct, "각 표준 ID 목록의 이웃한 두 출현만 직접 연결하세요."
expected_groups = sorted(list(expected_buckets.values()) + [[held_mention_id]])
assert identity_groups == expected_groups, "보류 기록을 포함한 전체 출현을 정확히 한 그룹씩 보존하세요."
expected_expanded = {pair_key(a, b) for group in expected_groups for a, b in combinations(group, 2)}
assert expanded_pairs == expected_expanded, "그룹 내부의 모든 쌍을 포함하고 없는 출현 ID를 넣지 마세요."
assert isinstance(symmetry_ok, bool) and symmetry_ok, "같은 쌍은 방향을 구분하지 않습니다."
print("✅ 통과!")


In [ ]:
# [제공코드] 서로 다른 확정 ID의 두 그룹을 일부러 합친 검사 입력입니다. 실제 연결 결과는 바꾸지 않습니다.
bucket_ids = sorted(id_buckets)
mixed_group = sorted(id_buckets[bucket_ids[0]] + id_buckets[bucket_ids[1]])
proposed_groups = [mixed_group] + [group[:] for group in identity_groups
    if not set(group) & set(mixed_group)]
pprint(proposed_groups[:2])


## 5. 서로 다른 표준 ID가 섞인 그룹을 보류합니다

**배경**: 이름 후보를 따라 묶은 그룹에 서로 다른 확정 ID가 섞이면 그대로 노드를 합칠 수 없습니다.  

**요구사항**  
- **partition_groups(groups, links)** 함수를 작성하세요. groups는 출현 ID 목록의 목록, links는 출현 ID를 표준 ID에 연결한 딕셔너리입니다. accepted는 ID 충돌이 없는 그룹, held는 서로 다른 확정 ID가 섞인 그룹입니다. 각 그룹에서 연결된 출현의 표준 ID만 집합으로 모으세요.
- **partition_groups** 는 서로 다른 표준 ID가 둘 이상이면 held, 0개 또는 1개면 accepted에 그 그룹을 복사하고 `(accepted, held)` 튜플을 반환합니다. 그룹 순서와 내부 순서를 유지하며 입력을 변경하지 마세요. 확정 ID 0개인 그룹도 보존하되 적재 승인을 뜻하지 않습니다.
- **accepted_groups** 와 **held_groups** 에 proposed_groups와 linked를 검사한 두 결과를 담으세요. 이후 문항은 검사용으로 합치기 전 identity_groups와 linked를 계속 사용합니다.

**확인 기준**: 인위적으로 섞은 mixed_group만 held_groups에 남습니다. 미확정 출현은 임의의 표준 ID에 연결하지 않습니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 그룹의 크기가 아니라 서로 다른 확정 ID의 수를 검사합니다.

세부구현:
1. 그룹의 출현 ID 중 연결이 있는 것만 조회합니다.
2. 서로 다른 표준 ID가 여러 개인 그룹을 분리합니다.
3. 각 그룹을 복사해 입력을 보존합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert held_groups == [mixed_group], "서로 다른 확정 ID가 섞인 그룹만 보류하세요."
assert accepted_groups == proposed_groups[1:], "나머지 그룹과 입력 순서를 보존하세요."
probe_groups = [["a", "b"], ["c", "d"], ["e"], ["f", "g"], ["h", "i"]]
probe_copy = [group[:] for group in probe_groups]
probe_links = {"a": "A", "b": "B", "c": "C", "d": "C", "f": "F", "h": "H", "i": "I"}
accepted_probe, held_probe = partition_groups(probe_groups, probe_links)
assert accepted_probe == [["c", "d"], ["e"], ["f", "g"]], "같은 ID·미확정 단독·한 ID와 미확정의 그룹은 분리 보존하세요."
assert held_probe == [["a", "b"], ["h", "i"]], "충돌 그룹을 빠짐없이 입력 순서로 보류하세요."
assert probe_groups == probe_copy, "입력 그룹을 변경하지 마세요."
for result_group in accepted_probe + held_probe:
    assert all(result_group is not source_group for source_group in probe_groups), "결과 그룹은 복사하세요."
print("✅ 통과!")


## 6. 양 끝이 확인된 트리플만 적재 대상으로 만듭니다

**배경**: 주어나 목적어의 표준 ID를 아직 확인하지 못한 트리플도 원래 관계와 근거를 보존해야 합니다.  

**요구사항**  
- **normalized_triples** 에는 양쪽 출현 ID가 모두 linked에 있는 task_triples 행만 담으세요. 양 끝의 출현 ID는 `triple_id`에 `:subject`와 `:object`를 붙여 만듭니다. 원래 행을 복사해 `subject_id`, `object_id`를 추가합니다.
- **held_triples** 에는 한쪽이라도 연결되지 않은 원래 트리플을 복사해 담으세요. subject_id와 object_id는 붙이지 않습니다. 두 목록 모두 입력 순서를 유지하세요.
- **normalized_triples** 와 **held_triples** 에서 원래 subject, relation, object와 출처 필드를 보존하세요.

**확인 기준**: 13개 트리플에 표준 ID가 붙고, 보류한 출현 기록이 포함된 1개 트리플은 held_triples에 남습니다. 두 목록의 triple_id를 합치면 원래 14개가 정확히 한 번씩 나옵니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 주어와 목적어의 출현 ID 두 개를 모두 확인한 뒤 분기합니다.

세부구현:
1. triple_id에서 양 끝의 출현 ID를 만듭니다.
2. 둘 다 연결되었으면 표준 ID를 붙입니다.
3. 하나라도 미해결이면 원래 행을 보류 목록에 남깁니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
expected_normalized = []
expected_held = []
for triple in original_triples:
    subject_id = triple["triple_id"] + ":subject"
    object_id = triple["triple_id"] + ":object"
    if subject_id in linked and object_id in linked:
        expected = dict(triple)
        expected.update(subject_id=linked[subject_id], object_id=linked[object_id])
        expected_normalized.append(expected)
    else:
        expected_held.append(dict(triple))
assert normalized_triples == expected_normalized, "원래 관계·출처와 정확한 양 끝의 표준 ID를 보존하세요."
assert held_triples == expected_held, "한쪽이라도 미해결인 원래 트리플을 별도로 남기세요."
assert task_triples == original_triples, "원래 추출 결과는 수정하지 마세요."
print("✅ 통과!")


### 보류한 트리플의 양 끝은 적재를 기다립니다

`t09:subject`가 보류되었으므로 `t09` 트리플 전체를 적재하지 않습니다.  
이미 ID를 확인한 `t09:object`도 이 트리플과 함께 적재를 기다립니다.  
다음 단계는 나머지 **13행의 양 끝 26건**으로 노드 기록을 만듭니다.  

앞의 15그룹은 평가할 출현 28건 전체를 묶은 결과입니다.  
아래의 13노드 기록은 지금 적재할 관계에 쓰이는 출현만 모은 결과입니다.  


## 7. 적재 노드와 관계에 원래 기록을 남깁니다

**배경**: 통합한 노드에서 원래 출현 기록을 추적하고 관계에서 원문 근거를 확인할 수 있어야 합니다.  

**요구사항**  
- **used_mentions** 에 normalized_triples의 양 끝에 해당하는 mentions 기록을 `mentions`의 원래 순서대로 담으세요. normalized_triples를 훑은 순서가 아니라 mentions를 걸러 낸 순서입니다.
- **node_rows** 에 used_mentions의 표준 ID마다 `standard_id`, `canonical_name`, `entity_type`, `aliases`, `mention_ids`, `mentions`의 여섯 키만 가진 딕셔너리를 하나씩 담고 standard_id 순서로 정렬하세요. 표준 이름·타입은 catalog에서, aliases는 그 표준 ID에 속한 출현들의 `name` 값을 중복 없이 정렬한 목록, mention_ids는 정렬한 출현 ID 목록, mentions는 해당 출현 기록을 입력 순서대로 복사한 목록입니다.
- **relationship_rows** 에는 normalized_triples의 입력 순서를 유지한 딕셔너리 리스트를 담으세요. 각 행을 복사합니다. 원래 모든 필드와 subject_id, object_id를 보존합니다.

**확인 기준**: 적재 대상은 13개 노드와 13개 관계이며, used_mentions의 26개 출현 ID가 노드에 정확히 한 번 들어갑니다. 관계의 두 표준 ID가 node_rows에 존재하고, 관계 이름과 원문 근거는 원래 트리플과 같습니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 실제 적재할 관계의 출현 기록만 모아 표준 ID별로 그룹화합니다.

세부구현:
1. 적재 대상 triple_id를 집합으로 만듭니다.
2. 해당 트리플의 출현 기록만 used_mentions에 남깁니다.
3. 표준 ID마다 원래 출현 기록을 보존한 노드를 만듭니다.
4. 정규화한 트리플 전체를 관계 자료로 복사합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
accepted_ids = {row["triple_id"] for row in normalized_triples}
expected_used = [row for row in mentions if row["triple_id"] in accepted_ids]
assert used_mentions == expected_used, "적재할 트리플의 주어와 목적어 출현만 입력 순서대로 보존하세요."
expected_node_ids = sorted({linked[row["mention_id"]] for row in expected_used})
assert [row["standard_id"] for row in node_rows] == expected_node_ids, "표준 ID마다 노드 하나를 정렬해 만드세요."
catalog_check = {row["standard_id"]: row for row in catalog}
all_mention_ids = []
for node in node_rows:
    standard_id = node["standard_id"]
    members = [row for row in expected_used if linked[row["mention_id"]] == standard_id]
    expected = {"standard_id": standard_id, "canonical_name": catalog_check[standard_id]["canonical_name"],
                "entity_type": catalog_check[standard_id]["entity_type"], "aliases": sorted({row["name"] for row in members}),
                "mention_ids": sorted(row["mention_id"] for row in members), "mentions": members}
    assert node == expected, "노드의 표준 ID와 타입, 원래 별칭과 출현 근거를 모두 보존하세요."
    all_mention_ids.extend(node["mention_ids"])
assert sorted(all_mention_ids) == sorted(row["mention_id"] for row in expected_used), "출현 기록이 누락되거나 중복 배정되지 않게 하세요."
assert relationship_rows == normalized_triples, "원래 관계 이름과 모든 출처 필드를 보존하세요."
node_ids = set(expected_node_ids)
assert all(row["subject_id"] in node_ids and row["object_id"] in node_ids for row in relationship_rows), "관계 양 끝은 실제 적재할 노드 ID여야 합니다."
print("✅ 통과!")


In [ ]:
# [제공코드] kg_gold.json: 교안과 과제 전체 출현의 정답 그룹입니다. 연결 계산이 끝난 뒤 평가에서만 읽습니다.
# 과제 범위 밖의 출현도 들어 있으므로 아래에서 evaluation_ids와 교집합해 범위를 맞춥니다.
gold_data = json.loads((data_dir / "kg_gold.json").read_text(encoding="utf-8"))
evaluation_ids = {row["mention_id"] for row in mentions}
print("평가 범위:", len(evaluation_ids), "출현 기록. 보류한 기록도 포함합니다.")


## 8. 보류한 기록까지 같은 범위에서 평가합니다

**배경**: ID 연결 단계에서 만든 파이썬 그룹을 평가하며, 보류 때문에 놓친 쌍도 포함합니다.  

**요구사항**  
**기본 평가**  

- **gold_pairs** 에 gold_data의 각 groups 항목에서 mention_ids를 evaluation_ids와 교집합한 뒤 만들 수 있는 모든 순서 없는 쌍을 집합으로 담으세요. pair_key를 사용합니다.
- **predicted_pairs** 에 identity_groups에서 만들 수 있는 모든 쌍을 집합으로 담으세요. 각 쌍은 pair_key(left_id, right_id)가 반환하는 정렬된 튜플로 표현하세요.
- **metrics** 에 `tp`, `fp`, `fn`, `precision`, `recall`, `f1`의 여섯 키만 가진 딕셔너리를 담으세요. precision은 TP/(TP+FP), recall은 TP/(TP+FN), f1은 2TP/(2TP+FP+FN)이며 분모가 0이면 0.0으로 계산합니다.
- **metrics** 와 골드에는 있지만 예측에는 없는 쌍을 출력하세요. evaluation_ids에서 pending을 빼지 마세요.
- **gold_id_by_mention** 딕셔너리에 gold_data의 groups에서 evaluation_ids에 포함된 각 mention_id를 그 그룹의 standard_id에 연결하세요.
- **id_correct** 에 mentions의 각 출현에서 linked의 ID가 gold_id_by_mention의 ID와 정확히 일치하는 건수를 정수로 담으세요. 보류한 출현의 예측 ID는 None으로 취급합니다.
- **id_errors** 에 일치하지 않은 출현마다 `(mention_id, 예측 ID 또는 None, 정답 ID)` 튜플을 mentions 입력 순서대로 담고, id_correct와 함께 출력하세요.
- **evaluate_pairs(predicted, gold)** 함수를 작성해 위의 여섯 키 지표 딕셔너리를 반환하세요. 인자는 정렬된 ID 튜플의 집합입니다. TP는 교집합, FP는 예측에만 있는 쌍, FN은 골드에만 있는 쌍의 수입니다. 앞의 **metrics** 도 이 함수로 구하세요. 입력을 변경하지 마세요.
- **evaluate_ids(rows, predictions, gold_ids)** 함수를 작성해 `(정확한 ID 일치 건수, 불일치 튜플 목록)`을 반환하세요. rows는 mention_id를 포함하는 딕셔너리 목록, predictions와 gold_ids는 출현 ID를 표준 ID에 연결한 딕셔너리입니다. gold_ids에는 모든 rows의 정답이 있습니다. rows 순서를 따르고 predictions에 없는 ID는 None입니다. **id_correct, id_errors** 도 이 함수로 구하고 입력을 변경하지 마세요.

**오류 사례 비교**  

- **과병합 사례**: 5번의 proposed_groups 안에서 가능한 모든 쌍을 **overmerge_pairs** 에 담고, 같은 gold_pairs로 평가한 결과를 **overmerge_metrics** 에 담으세요. 다른 개체를 합쳤을 때 FP와 precision이 어떻게 달라지는지 출력해 비교하세요.
- **틀린 ID 사례**: id_buckets의 표준 ID 키를 정렬한 첫 두 ID를 서로 맞바꾼 linked의 복사본을 **wrong_id_links** 에 담으세요. 나머지 ID와 보류 상태는 유지합니다. wrong_id_links에서 동일한 표준 ID를 가진 출현끼리의 모든 쌍을 **wrong_id_pairs** 에 담고 **wrong_id_metrics** 로 평가하세요. **wrong_id_correct, wrong_id_errors** 에 evaluate_ids로 평가한 결과를 담으세요. 원래 linked와 identity_groups는 바꾸지 마세요.

**빈 집합 검사**  

- **빈 쌍 사례**: gold_pairs에서 사전순 첫 쌍 하나만 담은 집합을 **one_pair** 로 만들고 **empty_metrics** 에 세 결과를 담으세요. 키 both_empty는 두 집합 모두 비었을 때, no_prediction은 예측이 비고 골드가 one_pair일 때, no_gold는 예측이 one_pair이고 골드가 비었을 때입니다. 모두 evaluate_pairs로 구하세요.
- **해석**: baseline(metrics), overmerge_metrics, wrong_id_metrics와 정확한 ID 일치 건수를 함께 출력하세요. 과병합과 틀린 ID를 각각 어떤 평가로 발견할 수 있는지, 빈 집합에서는 왜 분모를 따로 확인해야 하는지 설명하세요(표현 자유, 강사 확인).

**확인 기준**: 평가 범위는 보류한 기록을 포함한 28개 출현입니다. 골드 37쌍에 대해 TP 30, FP 0, FN 7입니다. ID가 정확한 출현은 27개이며, 보류한 1개는 id_errors에 남습니다. 그룹 구성이 같아도 다른 표준 ID를 붙였다면 ID 연결 오류입니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 전체 골드를 과제의 출현 범위로만 제한합니다. 연결 성공 여부로 범위를 줄이지 않습니다.

세부구현:
1. 골드 그룹과 evaluation_ids를 교집합합니다.
2. 골드와 예측 그룹 각각에서 모든 쌍을 만듭니다.
3. 교집합과 차집합으로 TP, FP, FN을 구합니다.
4. 분모가 0인 경우를 처리해 세 지표를 계산합니다.
5. 골드 그룹의 표준 ID를 각 출현 ID에 대응시킵니다.
6. 보류는 None으로 조회하고 ID 일치 건수와 불일치 튜플을 기록합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
expected_ids = {row["mention_id"] for row in mentions}
assert evaluation_ids == expected_ids, "보류한 출현 기록까지 평가 범위에 남기세요."
expected_gold = set()
for group in gold_data["groups"]:
    for left_id, right_id in combinations(sorted(set(group["mention_ids"]) & expected_ids), 2):
        expected_gold.add(pair_key(left_id, right_id))
expected_predicted = {pair_key(a, b) for group in identity_groups for a, b in combinations(group, 2)}
assert gold_pairs == expected_gold, "골드는 과제의 전체 출현 범위로만 제한하세요."
assert predicted_pairs == expected_predicted, "예측 그룹의 실제 출현 ID로 모든 쌍을 만드세요."
expected_tp = len(expected_predicted & expected_gold)
expected_fp = len(expected_predicted - expected_gold)
expected_fn = len(expected_gold - expected_predicted)
assert set(metrics) == {"tp", "fp", "fn", "precision", "recall", "f1"}, "지표의 여섯 키를 확인하세요."
assert (metrics["tp"], metrics["fp"], metrics["fn"]) == (expected_tp, expected_fp, expected_fn), "같은 골드를 기준으로 교집합과 차집합을 계산하세요."
denominators = {"precision": expected_tp + expected_fp, "recall": expected_tp + expected_fn, "f1": 2 * expected_tp + expected_fp + expected_fn}
for metric, denominator in denominators.items():
    numerator = 2 * expected_tp if metric == "f1" else expected_tp
    expected_value = numerator / denominator if denominator else 0.0
    assert abs(metrics[metric] - expected_value) < 1e-9, f"{metric}의 분모와 0 처리 조건을 확인하세요."
expected_id_mapping = {mention_id: group["standard_id"] for group in gold_data["groups"]
                       for mention_id in group["mention_ids"] if mention_id in expected_ids}
assert gold_id_by_mention == expected_id_mapping, "과제 범위의 각 출현 ID를 골드의 정확한 표준 ID에 연결하세요."
expected_id_errors = [(row["mention_id"], linked.get(row["mention_id"]), expected_id_mapping[row["mention_id"]])
                      for row in mentions if linked.get(row["mention_id"]) != expected_id_mapping[row["mention_id"]]]
assert type(id_correct) is int and id_correct == len(mentions) - len(expected_id_errors), "정확한 ID가 일치한 출현만 정수로 세세요."
assert id_errors == expected_id_errors, "보류한 출현은 None으로 표시하고 ID 불일치 튜플을 입력 순서대로 담으세요."
# 집합을 바꿔 같은 함수를 재호출해 FP가 없던 기본 입력에만 맞춘 오답을 구분합니다.
expected_overmerge = {pair_key(a, b) for group in proposed_groups for a, b in combinations(group, 2)}
assert overmerge_pairs == expected_overmerge, "충돌 검사용 그룹에서 가능한 모든 쌍을 만드세요."
swapped_keys = sorted(id_buckets)[:2]
swap_expected = {swapped_keys[0]: swapped_keys[1], swapped_keys[1]: swapped_keys[0]}
expected_wrong_links = {key: swap_expected.get(value, value) for key, value in linked.items()}
assert wrong_id_links == expected_wrong_links, "정렬한 첫 두 표준 ID만 서로 맞바꾸세요."
expected_wrong_pairs = {pair_key(a, b) for a, b in combinations(sorted(wrong_id_links), 2)
                        if wrong_id_links[a] == wrong_id_links[b]}
assert wrong_id_pairs == expected_wrong_pairs, "맞바꾼 ID를 기준으로 같은 그룹이 되는 쌍을 다시 구하세요."
assert wrong_id_metrics == evaluate_pairs(wrong_id_pairs, gold_pairs), "wrong_id_pairs를 evaluate_pairs로 평가한 결과를 담으세요."
assert wrong_id_pairs == expected_predicted, "ID 이름만 바뀌면 같은 그룹의 출현 쌍은 유지됩니다."
assert wrong_id_metrics == metrics, "동일 개체 쌍 평가는 ID 이름의 교환을 구별하지 않습니다."
assert one_pair == {min(expected_gold)}, "사전순 첫 골드 쌍 하나만 사용하세요."
assert set(empty_metrics) == {"both_empty", "no_prediction", "no_gold"}, "빈 쌍 사례의 세 키를 확인하세요."
pair_cases = [(predicted_pairs, gold_pairs, metrics),
              (overmerge_pairs, gold_pairs, overmerge_metrics),
              (wrong_id_pairs, gold_pairs, wrong_id_metrics),
              (set(), set(), empty_metrics["both_empty"]),
              (set(), one_pair, empty_metrics["no_prediction"]),
              (one_pair, set(), empty_metrics["no_gold"]),
              (one_pair, one_pair, None)]
for predicted, gold, stored in pair_cases:
    original_predicted, original_gold = set(predicted), set(gold)
    actual = evaluate_pairs(predicted, gold)
    tp_check, fp_check, fn_check = len(predicted & gold), len(predicted - gold), len(gold - predicted)
    expected_metrics = {"tp": tp_check, "fp": fp_check, "fn": fn_check}
    for key, numerator, denominator in [("precision", tp_check, tp_check + fp_check),
                                        ("recall", tp_check, tp_check + fn_check),
                                        ("f1", 2 * tp_check, 2 * tp_check + fp_check + fn_check)]:
        expected_metrics[key] = numerator / denominator if denominator else 0.0
    assert set(actual) == set(expected_metrics), "함수는 지표의 여섯 키만 반환해야 합니다."
    assert all(abs(actual[key] - value) < 1e-9 for key, value in expected_metrics.items()), "각 입력에서 교집합·차집합과 분모를 다시 계산하세요."
    if stored is not None:
        assert set(stored) == set(actual) and all(abs(stored[key] - value) < 1e-9 for key, value in actual.items()), "사례별 변수에도 실제 계산 결과를 담으세요."
    assert predicted == original_predicted and gold == original_gold, "평가하면서 입력 집합을 변경하지 마세요."
assert overmerge_metrics["fp"] > metrics["fp"], "다른 표준 개체를 합친 쌍은 FP로 검출되어야 합니다."
for rows, predictions in [(mentions, linked), (mentions, wrong_id_links),
                          (list(reversed(mentions)), {}), ([], {})]:
    rows_before = json.dumps(rows, ensure_ascii=False)
    predictions_before, gold_before = dict(predictions), dict(gold_id_by_mention)
    result = evaluate_ids(rows, predictions, gold_id_by_mention)
    expected_errors = [(row["mention_id"], predictions.get(row["mention_id"]), gold_id_by_mention[row["mention_id"]])
                       for row in rows if predictions.get(row["mention_id"]) != gold_id_by_mention[row["mention_id"]]]
    assert isinstance(result, tuple) and len(result) == 2, "ID 평가는 건수와 오류 목록의 튜플을 반환하세요."
    assert type(result[0]) is int and result == (len(rows) - len(expected_errors), expected_errors), "현재 인자의 ID와 입력 순서로 비교하세요."
    assert json.dumps(rows, ensure_ascii=False) == rows_before and predictions == predictions_before and gold_id_by_mention == gold_before, "ID 평가 입력을 변경하지 마세요."
assert (wrong_id_correct, wrong_id_errors) == evaluate_ids(mentions, wrong_id_links, gold_id_by_mention), "틀린 ID 사례의 일치 건수와 오류 목록을 기록하세요."
assert wrong_id_correct < id_correct, "그룹이 같아도 다른 표준 ID를 붙인 기록은 불일치로 세세요."
baseline_links = {key: value for key, value in expected_id_mapping.items() if key != held_mention_id}
baseline_buckets = {}
for mention_id, standard_id in baseline_links.items():
    baseline_buckets.setdefault(standard_id, []).append(mention_id)
baseline_groups = sorted([sorted(group) for group in baseline_buckets.values()] + [[held_mention_id]])
assert linked == baseline_links and identity_groups == baseline_groups, "오류 실험은 복사본에서 수행하고 원래 연결과 그룹을 유지하세요."
print("✅ 통과!")


## Neo4j 연결을 준비합니다

이제 실습용 Neo4j와 APOC가 필요합니다. 아래 연결 셀을 실행한 뒤 9~10번을 진행하세요.  


In [ ]:
# [제공코드] 노드 초기화와 적재에 사용할 실습 전용 Neo4j에 연결합니다.
import os
from urllib.parse import urlsplit
from dotenv import load_dotenv
from neo4j import GraphDatabase

# .env의 접속 주소와 계정 정보를 읽습니다. 값은 아래 환경 변수에서 가져옵니다.
load_dotenv(".env")
neo4j_uri = os.environ["NEO4J_URI"]
# driver는 여러 쿼리에서 재사용할 DB 연결 통로입니다. 계정 정보는 출력하지 않습니다.
driver = GraphDatabase.driver(
    neo4j_uri,
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]),
)
# 연결 객체 생성만으로 접속 성공이 보장되지 않으므로 지금 서버 접속을 확인합니다.
driver.verify_connectivity()

def run_cypher(query, **params):
    """값을 매개변수로 전달하고 Cypher 결과를 딕셔너리 목록으로 돌려줍니다."""
    # 쿼리마다 세션을 열고 with 블록이 끝나면 닫습니다. driver는 계속 재사용합니다.
    with driver.session() as session:
        # RETURN에서 붙인 별칭이 딕셔너리 키가 되어 파이썬에서 조회할 수 있습니다.
        return [record.data() for record in session.run(query, **params)]

# 주소에 계정 정보가 포함되어 있어도 호스트와 포트만 확인합니다.
connection_address = urlsplit(neo4j_uri)
print("Neo4j 연결 완료. 호스트:", connection_address.hostname, "/ 포트:", connection_address.port)


In [ ]:
# [제공코드] 개체 목록을 원래 타입과 표준 ID로 저장하는 함수를 준비합니다.
def put_standard_nodes(nodes):
    """원래 타입을 라벨로 써서 표준 ID별 노드를 저장합니다.

    Args:
        nodes (list[dict]): entity_type, standard_id와 노드 속성이 있는 기록 목록.

    Returns:
        list[dict]: [{'written': 처리한 노드 수}]. 기존 노드 갱신도 포함합니다.
    """
    return run_cypher("""
    // 목록의 표준 개체를 하나씩 적재합니다.
    UNWIND $rows AS row
    // entity_type이 Book이면 Book 라벨을 사용합니다. 같은 타입과 ID의 노드는 재사용합니다.
    MERGE (n:$(row.entity_type) {standard_id: row.standard_id})
    // 표준 이름과 원래 출현 기록을 함께 보존합니다.
    SET n += row
    // 처리한 표준 개체 수를 반환합니다.
    RETURN count(n) AS written
    """, rows=nodes)


# 예시 입력: '파이썬 입문'과 '파이썬 입문서'를 같은 초판으로 모은 노드입니다.
node_example_nodes = [{"standard_id": "book:B001", "canonical_name": "파이썬 입문 초판",
                  "entity_type": "Book", "aliases": ["파이썬 입문", "파이썬 입문서"]}]
# 예시 호출: 두 번 저장해도 같은 초판 노드를 재사용합니다.
# print(put_standard_nodes(node_example_nodes))
# print(put_standard_nodes(node_example_nodes))
# 예상 출력:
# [{'written': 1}]
# [{'written': 1}]
# written은 새로 만든 수가 아니라 처리한 기록 수입니다.


# 이미 저장된 주어와 목적어 노드를 찾아 원래 관계를 연결하는 함수입니다.
# 등장한 자리마다 만든 노드와 같은 개체를 하나로 모은 노드에 모두 사용할 수 있습니다.
# 관계 타입에는 원래 relation을 쓰고, triple_id로 서로 다른 추출 행을 구분합니다.
def put_relations(rows, node_key):
    """주어 노드에서 목적어 노드로 관계를 저장하고 같은 트리플은 중복 생성하지 않습니다.

    Args:
        rows (list[dict]): 관계와 양 끝 타입이 있는 트리플. 표준 ID 적재에는 양 끝 ID도 필요합니다.
        node_key (str): 노드를 찾을 ID 속성. occurrence_id 또는 standard_id.

    Returns:
        list[dict]: [{'written': 처리한 관계 수}]. 같은 양 끝·타입·triple_id의
            기존 관계는 속성을 갱신하며, 양 끝 노드가 없는 행은 제외합니다.
    """
    # ID 속성 이름만 쿼리에 직접 넣습니다. 라벨은 각 행의 원래 타입을 사용합니다.
    if node_key not in {"occurrence_id", "standard_id"}:
        raise ValueError("ID 속성은 occurrence_id 또는 standard_id를 사용하세요.")

    records = []
    for row in rows:
        # 이미 만든 노드의 저장 방식에 맞춰 찾을 ID를 정합니다. 새 ID를 부여하지 않습니다.
        if node_key == "occurrence_id":
            # 등장한 자리로 찾기: d01은 d01:subject와 d01:object 노드를 연결합니다.
            # 같은 책도 d01:object와 d02:object라는 별도 노드로 저장된 상태입니다.
            subject_key = row["triple_id"] + ":subject"
            object_key = row["triple_id"] + ":object"
        else:
            # 확정한 개체 ID로 찾기: d01의 person:001과 book:B001 노드를 연결합니다.
            # d02의 책도 book:B001이면 두 대출 관계가 같은 책 노드에 연결됩니다.
            subject_key = row["subject_id"]
            object_key = row["object_id"]
        records.append({"subject_key": subject_key, "object_key": object_key,
                        "properties": dict(row)})

    # 식별 속성에는 triple_id를 넣습니다. 같은 관계라도 출처 행이 다르면 보존합니다.
    query = f"""
    // 추출 행마다 원래 주어와 목적어의 식별자를 하나씩 처리합니다.
    UNWIND $rows AS item
    // 원래 주어와 목적어 타입을 라벨로 쓰고, 해당 ID의 기존 노드를 찾습니다.
    MATCH (s:$(item.properties.subject_type) {{{node_key}: item.subject_key}})
    MATCH (o:$(item.properties.object_type) {{{node_key}: item.object_key}})
    // $(...)는 각 행의 relation 값을 관계 타입으로 사용합니다.
    // 양 끝, 관계 타입과 triple_id가 같으면 기존 관계를 찾고, 없으면 만듭니다.
    MERGE (s)-[rel:$(item.properties.relation) {{triple_id: item.properties.triple_id}}]->(o)
    // 처음 저장하거나 다시 실행할 때 모두 원래 필드와 근거를 관계 속성에 기록합니다.
    SET rel += item.properties
    // 실제로 연결한 추출 행 수를 파이썬에서 확인할 수 있게 반환합니다.
    RETURN count(rel) AS written
    """
    return run_cypher(query, rows=records)


# 예시 입력: d01의 민수와 파이썬 입문 초판을 먼저 노드로 준비합니다.
relation_example_nodes = [{"standard_id": "person:001", "canonical_name": "민수", "entity_type": "Person"},
                 {"standard_id": "book:B001", "canonical_name": "파이썬 입문 초판", "entity_type": "Book"}]
relation_example_rows = [{"triple_id": "d01", "subject_id": "person:001", "relation": "빌림",
                 "subject_type": "Person", "object_id": "book:B001", "object_type": "Book",
                 "evidence": "민수는 김하나의 파이썬 입문 초판을 빌렸습니다."}]
# 예시 호출: 앞에서 준비한 put_standard_nodes로 양 끝 노드를 먼저 저장합니다.
# put_standard_nodes(relation_example_nodes)
# print(put_relations(relation_example_rows, "standard_id"))
# 예상 출력:
# [{'written': 1}]
# written은 새로 만든 수가 아니라 처리한 기록 수입니다.


## 9. 표준 노드와 원래 관계를 재실행해도 중복 없이 적재합니다

**배경**: 같은 추출 결과를 다시 적재해도 표준 노드와 원래 관계가 늘어나지 않아야 합니다.  

노드는 원래 타입인 `Document`, `ApiElement`, `Change`, `Issue`를 라벨로 사용합니다.  
제공 쿼리의 `$($row.entity_type)`은 노드 타입을, `$($row.relation)`은 관계 타입을 지정합니다.  
`graph_ids`는 준비 셀에서 만든 현재 자료의 표준 ID 목록입니다. 조회와 초기화는 이 범위에서만 수행합니다.  

**요구사항**  
- **load_graph()** 함수를 작성하세요. node_rows의 노드를 먼저 모두 적재하고, relationship_rows의 관계를 모두 적재합니다. 반환 값은 없습니다.
- **load_graph**의 노드 반복문에서는 각 노드의 mentions를 json.dumps로 직렬화하고, ensure_ascii=False를 지정하세요. Neo4j 속성에는 딕셔너리 목록 같은 중첩 구조를 그대로 넣을 수 없어 문자열로 바꿔 저장합니다. run_cypher에 node_upsert_query와 `row=해당 노드`, `mentions_json=직렬화한 문자열`을 전달하세요.
- **load_graph**의 관계 반복문에서는 해당 행의 relation이 allowed_relations에 포함되어 있는지 확인하세요. 포함되지 않으면 ValueError를 발생시키고, 포함되면 run_cypher에 relationship_upsert_query와 `row=해당 관계 행`을 전달하세요.
- **load_graph**는 allowed_relations를 호출할 때마다 다시 읽어야 합니다. 매개변수 기본값으로 묶어 두면 정의 시점의 값이 고정되어, 허용 목록이 바뀐 뒤 호출해도 예전 목록으로 검사합니다.
- **load_graph**를 정의한 뒤 아래 제공 실행 셀을 실행하세요. 직전 준비 단계에서 연결한 실습용 Neo4j에 같은 자료를 두 번 적재하고, 노드·관계 수와 원문 기록이 같은지 확인합니다.

**확인 기준**: 표준 노드 13개와 원래 관계 13개입니다. 두 번째 적재 뒤에도 같은 수와 같은 관계·근거가 남아야 합니다. 별도로 보류한 1개 트리플은 적재 대상에 포함하지 않습니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 교안에서 사용한 MERGE와 SET 쿼리는 제공되어 있습니다. 준비한 노드와 관계 자료를 빠짐없이 전달합니다.

세부구현:
1. node_rows를 순회하며 중첩된 출현 기록 목록을 JSON 문자열로 바꿉니다.
2. 원래 노드 행과 직렬화한 문자열을 노드 쿼리에 전달합니다.
3. relationship_rows를 순회하며 관계 타입을 확인하고 관계 쿼리에 전달합니다.
```

</details>


In [ ]:
# [제공코드] graph_ids는 현재 과제에서 적재할 표준 ID 목록입니다. 초기화와 조회에 사용합니다.
graph_ids = [row["standard_id"] for row in node_rows]

# 노드 라벨은 원래 entity_type, 관계 타입은 원래 relation을 사용합니다.
allowed_relations = {row["relation"] for row in original_triples}

# 노드는 표준 ID로 찾으므로 별칭이 늘어도 같은 노드를 사용합니다.
node_upsert_query = """
// 같은 표준 ID가 있으면 노드를 추가하지 않고 재사용합니다.
MERGE (n:$($row.entity_type) {standard_id: $row.standard_id})
// 표준 이름과 함께 원래 별칭, 출현 ID, 직렬화한 근거 기록도 저장합니다.
SET n.canonical_name = $row.canonical_name,
    n.entity_type = $row.entity_type,
    n.aliases = $row.aliases,
    n.mention_ids = $row.mention_ids,
    n.mentions_json = $mentions_json
// 어떤 표준 개체를 적재했는지 반환합니다.
RETURN n.standard_id AS standard_id
"""

# 원래 relation을 관계 타입으로 사용합니다. 같은 추출 행의 재실행은 triple_id로 찾습니다.
# $row에는 원래 표기, 원문과 저장 위치도 있으므로 생성과 재실행 때 모두 보존됩니다.
relationship_upsert_query = """
// 앞에서 적재한 두 표준 노드를 원래 주어와 목적어 방향으로 연결합니다.
MATCH (s:$($row.subject_type) {standard_id: $row.subject_id})
MATCH (o:$($row.object_type) {standard_id: $row.object_id})
// 같은 추출 행만 재사용하며, 다른 triple_id의 근거는 별도 관계로 남깁니다.
// $($row.relation)은 행에 기록된 관계 타입을 사용합니다(Neo4j 5.26 이상).
MERGE (s)-[rel:$($row.relation) {triple_id: $row.triple_id}]->(o)
// 생성과 재실행 모두 원래 필드와 근거를 관계 속성에 저장합니다.
SET rel += $row
// 실제로 적재한 추출 행을 확인할 수 있게 ID를 반환합니다.
RETURN rel.triple_id AS triple_id
"""
run_cypher("""
// 두 번 적재하기 전에 현재 과제의 표준 ID에 해당하는 노드만 초기화합니다.
MATCH (n)
WHERE n.standard_id IN $standard_ids DETACH DELETE n
""", standard_ids=graph_ids)
print("원래 타입과 표준 ID로 적재할 개체 수:", len(graph_ids))


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
# DB에서 다시 읽은 실제 노드와 관계를 검사합니다. 파이썬 입력 자료만 검사하지 않습니다.
def read_graph_snapshot():
    """현재 자료의 노드와 관계를 원래 타입과 출처 속성까지 조회합니다."""
    nodes = run_cypher("""
    // 적재 입력과 비교할 실제 노드 속성을 읽습니다.
    MATCH (n)
    WHERE n.standard_id IN $standard_ids
    RETURN n.standard_id AS standard_id, labels(n) AS labels, n.canonical_name AS canonical_name,
           n.entity_type AS entity_type, n.aliases AS aliases,
           n.mention_ids AS mention_ids, n.mentions_json AS mentions_json
    // 조회 순서가 아니라 내용 차이로 비교하도록 ID 순서를 고정합니다.
    ORDER BY standard_id
    """, standard_ids=graph_ids)
    for node in nodes:
        # 실제 라벨도 원래 타입과 같아야 합니다. 비교용 결과에는 원래 속성만 남깁니다.
        assert node.pop("labels") == [node["entity_type"]], "노드 라벨을 원래 entity_type으로 저장하세요."
        # 직렬화 방법의 공백 차이가 아닌 원래 출현 기록의 내용으로 비교합니다.
        node["mentions"] = json.loads(node.pop("mentions_json"))
    relationships = run_cypher("""
    // 관계 속성에 적힌 ID만 믿지 않고 실제 양 끝 노드의 ID를 읽습니다.
    MATCH (s)-[r]->(o)
    WHERE s.standard_id IN $standard_ids AND o.standard_id IN $standard_ids
    // 원래 방향, 타입, 모든 근거 속성을 적재 입력과 대조합니다.
    RETURN s.standard_id AS source_id, type(r) AS relation,
           o.standard_id AS target_id, properties(r) AS stored
    ORDER BY r.triple_id
    """, standard_ids=graph_ids)
    return {"nodes": nodes, "relationships": relationships}

expected_relationships = []
for row in sorted(relationship_rows, key=lambda row: row["triple_id"]):
    stored = dict(row)
    expected_relationships.append({"source_id": row["subject_id"], "relation": row["relation"],
                                   "target_id": row["object_id"], "stored": stored})
expected_snapshot = {"nodes": node_rows, "relationships": expected_relationships}

load_graph()
first_snapshot = read_graph_snapshot()
assert first_snapshot == expected_snapshot, "실제 DB에서 표준 ID, 원래 관계 타입과 출처·근거가 모두 보존되었는지 확인하세요."
load_graph()
second_snapshot = read_graph_snapshot()
assert second_snapshot == expected_snapshot, "재실행 후에도 노드와 관계가 중복되거나 원래 속성이 달라지면 안 됩니다."
assert second_snapshot == first_snapshot, "두 적재 결과가 같아야 합니다."

# 허용 관계가 없는 조건에서도 적재한다면 관계 타입 검사가 빠진 것입니다.
allowed_before_check = allowed_relations
allowed_relations = set()
rejected_relation = False
try:
    load_graph()
except ValueError:
    rejected_relation = True
finally:
    allowed_relations = allowed_before_check
assert rejected_relation, "허용하지 않은 관계는 ValueError로 거부하세요. allowed_relations를 매개변수 기본값으로 묶지 말고 호출할 때 읽으세요."
print("✅ 통과! 두 번 적재 후 표준 노드:", len(second_snapshot["nodes"]),
      "/ 원래 관계:", len(second_snapshot["relationships"]))


## 기존 출현 노드를 통합하는 실습 준비

앞 문항은 처음부터 표준 ID로 노드를 생성했습니다. 이번에는 이미 출현별 노드가 저장된 경우를 연습합니다.  
아래 제공 셀은 현재 자료의 표준 ID에 해당하는 노드와 그 노드에 연결된 모든 관계를 비우고,  
적재 대상 26개 출현을 각각 노드로 만든 뒤 원래 13개 관계를 연결합니다.  
보류한 트리플의 출현은 포함하지 않습니다. 반복하려면 이 준비 셀부터 다시 실행하세요.  


In [ ]:
# [제공코드] 원래 타입을 라벨로 쓰고, 같은 개체도 출현 ID마다 별도 노드로 준비합니다.
catalog_by_id = {row["standard_id"]: row for row in catalog}
occurrence_nodes = []
for row in used_mentions:
    standard_id = linked[row["mention_id"]]
    occurrence_nodes.append({
        "occurrence_id": row["mention_id"], "standard_id": standard_id,
        "canonical_name": catalog_by_id[standard_id]["canonical_name"],
        "entity_type": row["entity_type"], "aliases": [row["name"]],
        "mention_ids": [row["mention_id"]], "source_doc_ids": [row["source_doc_id"]],
    })

# 설정은 다음 답안에서 작성합니다. graph_ids 범위에서 같은 확정 ID끼리 모읍니다.
merge_nodes_query = """
// 현재 자료의 표준 ID에 해당하는 노드만 찾습니다.
MATCH (n)
WHERE n.standard_id IN $standard_ids
// 통합 후 남길 첫 노드를 일정하게 선택합니다.
WITH n ORDER BY n.occurrence_id
// 이름 유사도가 아니라 확정한 표준 ID가 같은 노드만 모읍니다.
WITH n.standard_id AS standard_id, collect(n) AS nodes
WHERE size(nodes) > 1
// 학생이 정한 정책으로 속성을 보존하며 관계를 통합 노드로 옮깁니다.
CALL apoc.refactor.mergeNodes(nodes, $config) YIELD node
// 실제로 통합한 그룹만 반환하므로 두 번째 실행 결과는 빈 목록입니다.
RETURN standard_id, node.mention_ids AS mention_ids
"""
run_cypher("""
// 앞 문항의 현재 자료 노드를 비우고, 출현별 저장 상태를 만듭니다.
MATCH (n)
WHERE n.standard_id IN $standard_ids DETACH DELETE n
""", standard_ids=graph_ids)
run_cypher("""
// 표준 ID가 같아도 우선은 출현마다 노드를 따로 만듭니다.
UNWIND $rows AS row
MERGE (n:$(row.entity_type) {occurrence_id: row.occurrence_id})
// 통합 뒤 원래 별칭과 출처를 확인할 수 있게 모든 준비 속성을 남깁니다.
SET n += row
""", rows=occurrence_nodes)
put_relations(normalized_triples, "occurrence_id")
before_nodes = run_cypher("""
// 각 출현이 통합 전에는 서로 다른 노드로 존재하는지 셉니다.
MATCH (n)
WHERE n.standard_id IN $standard_ids RETURN count(n) AS count
""", standard_ids=graph_ids)
assert before_nodes[0]["count"] == len(used_mentions), "준비 단계는 출현마다 노드를 하나씩 만듭니다."
print("통합 전 노드:", before_nodes[0]["count"])


## 10. APOC로 기존 노드를 통합하고 관계 이동을 확인합니다

**배경**: 출현별로 이미 생성한 중복 노드를 합쳐도 각 추출 관계와 근거를 보존해야 합니다.  

**요구사항**  
- **merge_config** 는 properties, mergeRels, singleElementAsArray 세 키만 가진 딕셔너리입니다. properties에는 aliases·mention_ids·source_doc_ids의 정책을 combine, 나머지 속성에 적용할 `.*` 정책을 discard로 지정합니다. properties에는 위 네 키만 사용합니다. mergeRels는 False, singleElementAsArray는 True로 지정하세요.
- **merge_duplicates()** 함수를 작성하세요. run_cypher에 제공된 merge_nodes_query, `config=merge_config`, `standard_ids=graph_ids`를 전달하고 조회 결과를 그대로 반환하세요. 함수 정의만 하고 실행은 아래 제공 자가채점 셀에서 수행합니다.
- **merge_config** 와 **merge_duplicates** 의 결과로 같은 표준 ID의 노드는 하나가 되고 원래 타입 라벨과 13개 관계의 방향·타입·모든 속성이 유지되어야 합니다. 아래 실제 DB 검사에서 출현 소속까지 확인하세요.

**확인 기준**: 설정은 DB 없이 검사합니다. 실제 DB에서는 26개 출현 노드가 13개 표준 노드로 통합되고, 관계 13개가 각각 남습니다. 두 번 통합해도 결과가 같습니다.  

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 속성 병합 정책과 관계 보존 옵션을 사용합니다.

세부구현:
1. 여러 출현의 값을 보존할 목록 속성에 결합 정책을 지정합니다.
2. 관계를 합치지 않고 한 값도 목록으로 남기는 옵션을 지정합니다.
3. 제공 쿼리에 통합 설정을 전달합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert set(merge_config) == {"properties", "mergeRels", "singleElementAsArray"}, "설정의 세 키를 확인하세요."
assert merge_config["properties"] == {"aliases": "combine", "mention_ids": "combine", "source_doc_ids": "combine", ".*": "discard"}, "목록 속성은 결합하고 나머지는 첫 값을 유지하세요."
assert merge_config["mergeRels"] is False, "원래 추출 행별 관계를 보존하세요."
assert merge_config["singleElementAsArray"] is True, "값 하나도 목록 형식을 유지하세요."
assert callable(merge_duplicates), "merge_duplicates 함수를 정의하세요."
print("✅ 설정 통과! 실제 노드 통합은 다음 DB 검사에서 확인합니다.")


In [ ]:
# [자가채점]
# 실제 관계의 양 끝을 DB 노드 식별자로 비교하여 통합하지 않은 오답을 구분합니다.
expected_by_id = {row["triple_id"]: row for row in normalized_triples}
expected_members = {row["standard_id"]: set(row["mention_ids"]) for row in node_rows}
# 출현 ID로 바로 조회하도록 준비해 노드마다 원본 목록 전체를 검색하지 않습니다.
used_mention_by_id = {row["mention_id"]: row for row in used_mentions}
for repeat in range(2):
    # 첫 통합은 중복 그룹을, 두 번째 통합은 처리할 그룹이 없는 빈 목록을 반환합니다.
    merge_result = merge_duplicates()
    assert isinstance(merge_result, list), "쿼리의 조회 결과 목록을 반환하세요."
    expected_merged = {}
    if repeat == 0:
        expected_merged = {key: members for key, members in expected_members.items() if len(members) > 1}
    assert all(set(row) == {"standard_id", "mention_ids"} for row in merge_result), "조회 결과의 두 필드를 그대로 반환하세요."
    returned_members = {row["standard_id"]: set(row["mention_ids"]) for row in merge_result}
    assert len(merge_result) == len(expected_merged) and returned_members == expected_merged, "실제로 통합한 그룹의 조회 결과를 반환하세요."
    actual_nodes = run_cypher("""
    // 표준 ID만 같고 실제로는 별도 노드인 경우를 구분하려고 elementId도 읽습니다.
    MATCH (n)
    WHERE n.standard_id IN $standard_ids
    RETURN elementId(n) AS node_id, n.standard_id AS standard_id, labels(n) AS labels,
           n.mention_ids AS mention_ids, n.aliases AS aliases,
           n.source_doc_ids AS source_doc_ids
    """, standard_ids=graph_ids)
    actual_edges = run_cypher("""
    // 관계가 실제로 옮겨진 시작점과 도착점을 따라 출현 소속을 다시 계산합니다.
    MATCH (s)-[r]->(o)
    WHERE s.standard_id IN $standard_ids AND o.standard_id IN $standard_ids
    // 원래 관계 타입과 모든 근거 속성이 보존됐는지도 함께 검사합니다.
    RETURN elementId(s) AS source_node, elementId(o) AS target_node,
           s.standard_id AS subject_id, o.standard_id AS object_id,
           type(r) AS relation, properties(r) AS stored
    """, standard_ids=graph_ids)
    assert len(actual_nodes) == len(expected_members), "같은 표준 ID의 실제 노드를 하나로 합치세요."
    assert {row["standard_id"] for row in actual_nodes} == set(expected_members), "표준 ID를 보존하세요."
    actual_members = {}
    seen_triples = []
    for edge in actual_edges:
        triple_id = edge["stored"]["triple_id"]
        assert triple_id in expected_by_id, "원래 자료에 없는 관계를 만들지 마세요."
        original = expected_by_id[triple_id]
        seen_triples.append(triple_id)
        expected_properties = dict(original)
        assert edge["stored"] == expected_properties, "각 추출 관계의 모든 속성과 근거를 보존하세요."
        assert edge["relation"] == original["relation"], "원래 관계 타입을 유지하세요."
        assert edge["subject_id"] == original["subject_id"] and edge["object_id"] == original["object_id"], "관계의 방향과 표준 ID를 확인하세요."
        actual_members.setdefault(edge["source_node"], set()).add(triple_id + ":subject")
        actual_members.setdefault(edge["target_node"], set()).add(triple_id + ":object")
    assert len(seen_triples) == len(set(seen_triples)) == len(expected_by_id), "관계가 누락되거나 합쳐지거나 중복되면 안 됩니다."
    assert set(seen_triples) == set(expected_by_id), "원래 모든 추출 행을 보존하세요."
    for node in actual_nodes:
        expected_type = catalog_by_id[node["standard_id"]]["entity_type"]
        assert node["labels"] == [expected_type], "통합 후에도 원래 타입만 라벨로 남아야 합니다."
        members = expected_members[node["standard_id"]]
        assert actual_members.get(node["node_id"]) == members, "실제 관계의 양 끝이 올바른 통합 노드로 이동해야 합니다."
        assert len(node["mention_ids"]) == len(set(node["mention_ids"])) and set(node["mention_ids"]) == members, "출현 ID를 누락·중복 없이 보존하세요."
        originals = [used_mention_by_id[mention_id] for mention_id in members]
        assert isinstance(node["aliases"], list) and set(node["aliases"]) == {row["name"] for row in originals}, "별칭은 원래 표기의 목록으로 보존하세요."
        assert isinstance(node["source_doc_ids"], list) and set(node["source_doc_ids"]) == {row["source_doc_id"] for row in originals}, "출처 문서는 목록으로 보존하세요."
print("✅ 통과! 두 번 통합 후 실제 노드와 관계, 출현 소속, 원문 근거가 같습니다.")
